# Notebook 02 - Missing Data & Outlier Analysis

**Goal:** Build a complete picture of data quality before any imputation.

1. Time index audit - align all datasets to a common hourly UTC spine
2. Missing value structure - heatmaps, run-length distributions
3. Per-feature deep dive - generation by PSR type, cross-border by direction
4. Cross-border zero-vs-NaN diagnostic
5. Missing data mechanism - MCAR / MAR / MNAR indicators
6. Outlier identification with physical bounds
7. Summary + Glocal-IB input preparation

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RAW = "../data/raw"
PROCESSED = "../data/processed"
os.makedirs(PROCESSED, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})
print("Imports OK")

In [ ]:
prices      = pd.read_parquet(f"{RAW}/entsoe_prices_2019_2025.parquet")
load        = pd.read_parquet(f"{RAW}/entsoe_load_2019_2025.parquet")
generation  = pd.read_parquet(f"{RAW}/entsoe_generation_2019_2025.parquet")
crossborder = pd.read_parquet(f"{RAW}/entsoe_crossborder_2019_2025.parquet")
weather     = pd.read_parquet(f"{RAW}/weather_2019_2025.parquet")
fuels       = pd.read_parquet(f"{RAW}/fuels_2019_2025.parquet")

for name, df in [("prices", prices), ("load", load), ("generation", generation),
                 ("crossborder", crossborder), ("fuels", fuels)]:
    print(f"{name:15s}  shape={str(df.shape):<20}  index={type(df.index).__name__}")
print(f"\nweather  shape={str(weather.shape):<20}  levels={weather.index.names}")

## 1. Time Index Audit

In [ ]:
SPINE = pd.date_range("2019-01-01", "2025-12-31 23:00", freq="1h", tz="UTC")
print(f"Full spine: {len(SPINE):,} hours  ({SPINE[0]} -> {SPINE[-1]})")

def index_report(name, idx):
    if not isinstance(idx, pd.DatetimeIndex):
        idx = pd.DatetimeIndex(idx)
    idx_utc = idx.tz_convert("UTC") if idx.tzinfo else idx.tz_localize("UTC")
    idx_utc = idx_utc.floor("h")
    idx_utc = idx_utc[(idx_utc >= SPINE[0]) & (idx_utc <= SPINE[-1])]
    missing_hours = SPINE.difference(idx_utc)
    dup = idx_utc.duplicated().sum()
    print(f"{name:18s}  {len(idx_utc):>8,} rows  {len(missing_hours):>6,} missing hrs  {dup:>4} dupes")
    return missing_hours

print()
gaps = {}
for name, df in [("prices", prices), ("load", load), ("generation", generation), ("crossborder", crossborder)]:
    if isinstance(df.index, pd.DatetimeIndex):
        gaps[name] = index_report(name, df.index)

# Detect which MultiIndex level holds city names (strings) vs datetimes
city_level_idx = next(
    lvl for lvl in range(weather.index.nlevels)
    if not pd.api.types.is_datetime64_any_dtype(weather.index.get_level_values(lvl))
)
first_city = weather.index.get_level_values(city_level_idx).unique()[0]
weather_dt_idx = weather.xs(first_city, level=city_level_idx).index
gaps["weather"] = index_report(f"weather ({first_city})", weather_dt_idx)

In [ ]:
# Missing-VALUE hours per month. Generation and cross-border have (near-)complete
# hourly timestamps, so their data-quality problem is NaN *values* inside present
# rows, not absent rows. We reindex each source onto the SPINE and count hours
# that carry at least one NaN value, per month. (The previous version plotted
# missing *timestamps* via SPINE.difference(index), which is ~empty for these
# sources -> empty bars.)
fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
for ax, name, df in zip(axes, ["generation", "crossborder"], [generation, crossborder]):
    idx = df.index
    idx = idx.tz_convert("UTC") if idx.tzinfo else idx.tz_localize("UTC")
    s = df.copy()
    s.index = idx.floor("h")
    s = s[~s.index.duplicated(keep="last")].reindex(SPINE)
    monthly = s.isna().any(axis=1).resample("ME").sum()
    ax.bar(monthly.index, monthly.values, width=20, color="steelblue", alpha=0.7)
    ax.set_title(f"{name} - missing VALUE hours per month (rows with >=1 NaN)")
    ax.set_ylabel("# missing hrs")
plt.tight_layout()
plt.show()


## 2. Missing Value Structure

In [ ]:
def to_spine(df):
    idx = df.index
    if not isinstance(idx, pd.DatetimeIndex):
        return df
    idx_utc = idx.tz_convert("UTC") if idx.tzinfo else idx.tz_localize("UTC")
    df2 = df.copy()
    df2.index = idx_utc.floor("h")
    df2 = df2[~df2.index.duplicated(keep="last")]
    return df2.reindex(SPINE)

prices_s      = to_spine(prices)
load_s        = to_spine(load)
generation_s  = to_spine(generation)
crossborder_s = to_spine(crossborder)

for name, df in [("prices", prices_s), ("load", load_s),
                 ("generation", generation_s), ("crossborder", crossborder_s)]:
    print(f"{name:15s}  {df.shape}  overall NaN={df.isna().mean().mean():.1%}")

In [ ]:
def nan_heatmap(df, title, figsize=(14, 5)):
    monthly = df.resample("ME").apply(lambda x: x.isna().mean())
    monthly.index = monthly.index.to_period("M").astype(str)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(monthly.T, ax=ax, cmap="Reds", vmin=0, vmax=1,
                linewidths=0.3, cbar_kws={"label": "NaN rate"})
    ticks = range(0, len(monthly.index), 6)
    ax.set_xticks(ticks)
    ax.set_xticklabels([monthly.index[i] for i in ticks], rotation=45, ha="right")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

nan_heatmap(generation_s,  "Generation - NaN rate by feature x month")
nan_heatmap(crossborder_s, "Cross-border - NaN rate by feature x month", figsize=(14, 3))

In [ ]:
def run_lengths(series):
    runs, count = [], 0
    for v in series.isna():
        if v:
            count += 1
        elif count:
            runs.append(count)
            count = 0
    if count:
        runs.append(count)
    return np.array(runs)

print(f"{'Column':<25} {'NaN%':>6} {'N runs':>8} {'Median':>8} {'Max':>8} {'>168h%':>8}")
print("-" * 70)
for col in list(generation_s.columns) + list(crossborder_s.columns):
    src = generation_s if col in generation_s.columns else crossborder_s
    nan_rate = src[col].isna().mean()
    if nan_rate < 0.005:
        continue
    r = run_lengths(src[col])
    if len(r) == 0:
        continue
    pct_week = (r > 168).mean()
    print(f"{col:<25} {nan_rate:>6.1%} {len(r):>8} {np.median(r):>8.0f} {r.max():>8} {pct_week:>8.1%}")

In [ ]:
worst_gen = generation_s.isna().mean().nlargest(4).index.tolist()
worst_cb  = crossborder_s.isna().mean().nlargest(3).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, col in enumerate(worst_gen):
    r = run_lengths(generation_s[col])
    axes[0, i].hist(r, bins=40, color="steelblue", alpha=0.8)
    axes[0, i].set_title(f"{col}\n(NaN={generation_s[col].isna().mean():.0%})", fontsize=8)
    axes[0, i].set_yscale("log")
    axes[0, i].set_xlabel("Run length (hrs)")

for i, col in enumerate(worst_cb):
    r = run_lengths(crossborder_s[col])
    axes[1, i].hist(r, bins=40, color="tomato", alpha=0.8)
    axes[1, i].set_title(f"{col}\n(NaN={crossborder_s[col].isna().mean():.0%})", fontsize=8)
    axes[1, i].set_yscale("log")
    axes[1, i].set_xlabel("Run length (hrs)")

for ax in axes[1, len(worst_cb):]:
    ax.set_visible(False)

plt.suptitle("NaN run-length histograms (log y)", y=1.01)
plt.tight_layout()
plt.show()

## 3. Per-Feature Deep Dive

In [ ]:
gen_yearly_nan = (
    generation_s
    .assign(year=generation_s.index.year)
    .groupby("year")
    .apply(lambda g: g.drop(columns="year").isna().mean())
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(gen_yearly_nan.T, ax=ax, cmap="Reds", vmin=0, vmax=1,
            annot=True, fmt=".0%", linewidths=0.3, cbar_kws={"label": "NaN rate"})
ax.set_title("Generation - NaN rate per source per year")
plt.tight_layout()
plt.show()

In [ ]:
cb_yearly_nan = (
    crossborder_s
    .assign(year=crossborder_s.index.year)
    .groupby("year")
    .apply(lambda g: g.drop(columns="year").isna().mean())
)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(cb_yearly_nan.T, ax=ax, cmap="Reds", vmin=0, vmax=1,
            annot=True, fmt=".0%", linewidths=0.3, cbar_kws={"label": "NaN rate"})
ax.set_title("Cross-border - NaN rate per direction per year")
plt.tight_layout()
plt.show()

## 4. Cross-Border Zero vs NaN Diagnostic

Does `NaN` in the France columns mean **zero flow** or **reporting failure**?  
We test whether FR gaps correlate with overall grid activity on the AT/CH borders.

In [ ]:
cb = crossborder_s.copy()
at_ch_cols = [c for c in cb.columns if ("AT" in c or "CH" in c) and "net" not in c]
cb["atch_total"] = cb[at_ch_cols].abs().sum(axis=1, skipna=True)

fr_nan = cb["FR_to_DELU"].isna()
miss_activity = cb.loc[fr_nan, "atch_total"]
pres_activity = cb.loc[~fr_nan, "atch_total"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(miss_activity.dropna(), bins=60, alpha=0.6, color="tomato",   label=f"FR NaN  (n={fr_nan.sum():,})")
ax.hist(pres_activity.dropna(), bins=60, alpha=0.6, color="steelblue", label=f"FR observed (n={(~fr_nan).sum():,})")
ax.set_xlabel("AT+CH total flow (MW)")
ax.set_ylabel("Hours")
ax.legend()
ax.set_title("AT/CH grid activity when FR is missing vs. observed")
plt.tight_layout()
plt.show()

u, p = stats.mannwhitneyu(miss_activity.dropna(), pres_activity.dropna(), alternative="two-sided")
print(f"Mann-Whitney p={p:.2e}")
if p < 0.05:
    print("=> FR gaps correlate with grid activity -> structural reporting gap, not zero-flow")
else:
    print("=> FR gaps look independent of grid activity -> zero-fill may be appropriate")

In [ ]:
cb_cols = [c for c in ["FR_to_DELU", "DELU_to_FR", "AT_to_DELU", "DELU_to_AT", "CH_to_DELU", "DELU_to_CH"]
           if c in crossborder_s.columns]
monthly_avail = crossborder_s[cb_cols].resample("ME").apply(lambda x: x.notna().mean())
monthly_avail.index = monthly_avail.index.to_period("M").astype(str)

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(monthly_avail.T, ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=0.3, cbar_kws={"label": "% available"})
ticks = range(0, len(monthly_avail.index), 6)
ax.set_xticks(ticks)
ax.set_xticklabels([monthly_avail.index[i] for i in ticks], rotation=45, ha="right")
ax.set_title("Cross-border availability by month")
plt.tight_layout()
plt.show()

## 5. Missing Data Mechanism

**MNAR check:** does gap probability correlate with what the value would have been?  
We proxy using price (marginal cost signal) - if low-merit generators go offline at high-price hours and aren't reported, that's MNAR.

In [ ]:
price_col = prices_s.columns[0]
price_series = prices_s[price_col]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
candidates = worst_gen + [c for c in worst_cb if c in crossborder_s.columns][:2]
for ax, col in zip(axes.flat, candidates[:6]):
    src = generation_s if col in generation_s.columns else crossborder_s
    miss_flag = src[col].isna().astype(int)
    combined = pd.concat([miss_flag.rename("missing"), price_series.rename("price")], axis=1).dropna()
    try:
        bins = pd.qcut(combined["price"], q=10, duplicates="drop")
        miss_by_price = combined.groupby(bins, observed=True)["missing"].mean()
        ax.bar(range(len(miss_by_price)), miss_by_price.values, color="steelblue", alpha=0.8)
        ax.set_xticks(range(len(miss_by_price)))
        ax.set_xticklabels([f"{b.mid:.0f}" for b in miss_by_price.index], rotation=45, ha="right", fontsize=7)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha="center")
    ax.set_title(f"{col}\nP(missing) by price decile", fontsize=8)
    ax.set_ylabel("P(NaN)")

plt.suptitle("MNAR check: does gap probability vary with electricity price?", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
miss_df = pd.concat([
    generation_s[worst_gen].isna().add_prefix("gen_"),
    crossborder_s[[c for c in ["FR_to_DELU", "DELU_to_FR", "AT_to_DELU", "DELU_to_AT"]
                   if c in crossborder_s.columns]].isna().add_prefix("cb_"),
], axis=1).astype(float)

corr = miss_df.corr()
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=ax, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", square=True, linewidths=0.3)
ax.set_title("Missingness correlation - do gaps co-occur?")
plt.tight_layout()
plt.show()

## 6. Outlier Identification

In [ ]:
price_col = prices_s.columns[0]
load_col  = load_s.columns[0]

for name, series, lo, hi in [
    ("Price (EUR/MWh)",  prices_s[price_col], -500,    4000),
    ("Load (MW)",        load_s[load_col],    20_000, 100_000),
]:
    out = series[(series < lo) | (series > hi)]
    print(f"{name}: {len(out)} values outside [{lo}, {hi}]")
    if len(out) > 0:
        print(f"  range=[{out.min():.1f}, {out.max():.1f}]  first 3:{out.head(3).to_dict()}")

In [ ]:
neg = prices_s[prices_s[price_col] < 0]
print(f"Negative prices: {len(neg):,} / {len(prices_s):,} = {len(neg)/len(prices_s):.1%}")
print(f"  min={neg[price_col].min():.1f}  mean={neg[price_col].mean():.1f}  p5={neg[price_col].quantile(0.05):.1f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(neg[price_col], bins=50, color="tomato", alpha=0.8)
axes[0].set_title("Distribution of negative prices")
axes[0].set_xlabel("Price (EUR/MWh)")

neg_by_hour = neg.assign(hour=neg.index.hour).groupby("hour")[price_col].count()
axes[1].bar(neg_by_hour.index, neg_by_hour.values, color="tomato", alpha=0.8)
axes[1].set_title("Negative prices by hour of day (UTC)")
axes[1].set_xlabel("Hour")

neg_by_year = neg.assign(year=neg.index.year).groupby("year")[price_col].count()
axes[2].bar(neg_by_year.index, neg_by_year.values, color="tomato", alpha=0.8)
axes[2].set_title("Negative prices by year")
axes[2].set_xlabel("Year")

plt.tight_layout()
plt.show()

In [ ]:
# New Year load dip - overlay each year
load_series = load_s[load_col].dropna()
fig, ax = plt.subplots(figsize=(14, 4))
for year in range(2019, 2026):
    end_year = min(year + 1, 2025)
    end_date = f"{end_year}-01-08" if year < 2025 else f"{year}-12-31 23:00"
    window = load_series[(load_series.index >= f"{year}-12-24") & (load_series.index <= end_date)]
    if len(window) > 24:
        ax.plot(np.arange(len(window)), window.values / 1000, alpha=0.7, label=str(year))
ax.set_title("Load around Christmas/New Year (overlaid per year)")
ax.set_xlabel("Hours from Dec 24")
ax.set_ylabel("Load (GW)")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

print("\nJan 1 avg load vs. yearly median (ratio):")
yearly_med = load_series.groupby(load_series.index.year).median()
for year in range(2019, 2026):
    jan1 = load_series[load_series.index.normalize() == pd.Timestamp(f"{year}-01-01", tz="UTC")]
    if year in yearly_med.index and len(jan1) > 0:
        print(f"  {year}: {jan1.mean()/1e3:.1f} GW vs {yearly_med[year]/1e3:.1f} GW median  (ratio={jan1.mean()/yearly_med[year]:.2f})")

In [ ]:
# Generation outliers via IQR fence
print(f"{'Column':<25} {'Outliers':>9} {'Min':>10} {'Max':>10}")
print("-" * 58)
for col in generation_s.columns:
    s = generation_s[col].dropna()
    if len(s) < 100:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    out = s[(s < q1 - 3*iqr) | (s > q3 + 3*iqr)]
    if len(out) > 0:
        print(f"{col:<25} {len(out):>9,} {out.min():>10.0f} {out.max():>10.0f}")

In [ ]:
# Fuel prices: forward-fill daily -> hourly
fuels_idx = fuels.index
if hasattr(fuels_idx, 'tz') and fuels_idx.tz is None:
    fuels_idx = fuels_idx.tz_localize("UTC")
fuels_h = fuels.copy()
fuels_h.index = fuels_idx
fuels_h = fuels_h.reindex(SPINE).ffill()

print(f"Fuels after ffill: {fuels_h.shape}  remaining NaN: {fuels_h.isna().sum().to_dict()}")

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for ax, col in zip(axes, fuels_h.columns):
    ax.plot(fuels_h.index, fuels_h[col], lw=0.5)
    ax.set_title(col)
    ax.set_ylabel("Price")
plt.tight_layout()
plt.show()

## 7. Summary & Glocal-IB Input Preparation

Build the combined feature matrix and missing mask for Glocal-IB.  
**Interface expected:** `[B, T, N]` float32 tensor with `NaN` for missing values.

In [ ]:
summary_rows = []
for col in list(generation_s.columns) + list(crossborder_s.columns):
    src = generation_s if col in generation_s.columns else crossborder_s
    nan_rate = src[col].isna().mean()
    if nan_rate == 0:
        continue
    r = run_lengths(src[col])
    max_run = r.max() if len(r) > 0 else 0
    verdict = "impute" if nan_rate < 0.65 else "review"
    dataset = "generation" if col in generation_s.columns else "crossborder"
    summary_rows.append({"dataset": dataset, "column": col,
                         "nan_rate": f"{nan_rate:.1%}", "max_run_hrs": int(max_run), "verdict": verdict})

summary_df = pd.DataFrame(summary_rows).sort_values(["dataset", "nan_rate"], ascending=[True, False])
print(summary_df.to_string(index=False))

In [ ]:
# Flatten weather MultiIndex - detect city level by dtype (strings, not datetimes)
city_level_idx = next(
    lvl for lvl in range(weather.index.nlevels)
    if not pd.api.types.is_datetime64_any_dtype(weather.index.get_level_values(lvl))
)
city_level = weather.index.names[city_level_idx]
cities = weather.index.get_level_values(city_level_idx).unique()
weather_parts = []
for city in cities:
    cdf = weather.xs(city, level=city_level_idx).copy()
    cdf.columns = [f"{city.lower()}_{c}" for c in cdf.columns]
    weather_parts.append(cdf)

weather_flat = pd.concat(weather_parts, axis=1)
if weather_flat.index.tzinfo is None:
    weather_flat.index = weather_flat.index.tz_localize("UTC")
weather_flat = weather_flat.reindex(SPINE)
print(f"Weather flat: {weather_flat.shape}  NaN={weather_flat.isna().mean().mean():.1%}")

In [ ]:
price_renamed = prices_s.rename(columns={prices_s.columns[0]: "price"})
load_renamed  = load_s.rename(columns={load_s.columns[0]: "load"})
cb_raw_cols   = [c for c in crossborder_s.columns if "net_" not in c]

combined = pd.concat([
    price_renamed[["price"]],
    load_renamed[["load"]],
    generation_s,
    crossborder_s[cb_raw_cols],
    weather_flat,
    fuels_h,
], axis=1)

combined = combined[(combined.index >= SPINE[0]) & (combined.index <= SPINE[-1])]
missing_mask = combined.isna().astype("uint8")

print(f"Combined matrix : {combined.shape}")
print(f"Date range      : {combined.index[0]} -> {combined.index[-1]}")
print(f"Overall NaN     : {combined.isna().mean().mean():.1%}")
print(f"Columns with NaN: {(combined.isna().any()).sum()} / {combined.shape[1]}")
print(f"\nTop 10 NaN columns:")
print(combined.isna().mean().nlargest(10).apply(lambda x: f"{x:.1%}").to_string())

In [ ]:
combined.to_parquet(f"{PROCESSED}/emis_raw.parquet")
missing_mask.to_parquet(f"{PROCESSED}/emis_mask.parquet")
print(f"Saved  {PROCESSED}/emis_raw.parquet   {combined.shape}")
print(f"Saved  {PROCESSED}/emis_mask.parquet  {missing_mask.shape}")
print("\nNext: src/preprocessing/impute.py - adapt GlocalIB/ to this matrix.")

In [ ]:
# ── SUMMARY REPORT ──────────────────────────────────────────────────────────
# Run this cell last. Its output is a self-contained snapshot of the dataset
# state that can be pasted directly into a new conversation for full context.

import textwrap, io
out = io.StringIO()

def p(*args, **kw):
    print(*args, **kw, file=out)

p("=" * 72)
p("EMIS DATA QUALITY SNAPSHOT")
p(f"Generated: {pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d %H:%M UTC')}")
p("=" * 72)

# 1. Spine
p(f"\n[SPINE]  {len(SPINE):,} hourly UTC slots  "
  f"{SPINE[0].date()} -> {SPINE[-1].date()}")

# 2. Raw dataset shapes & index resolution
p("\n[RAW DATASETS]")
raw_info = [
    ("prices",      prices,      "15-min, deduplicated to 1h"),
    ("load",        load,        "15-min, deduplicated to 1h"),
    ("generation",  generation,  "15-min, deduplicated to 1h"),
    ("crossborder", crossborder, "1h native"),
    ("weather",     weather,     "1h native, MultiIndex (datetime x city)"),
    ("fuels",       fuels,       "daily, forward-filled to 1h"),
]
for name, df, note in raw_info:
    p(f"  {name:<15s} {str(df.shape):<22}  {note}")

# 3. Spine-aligned NaN rates
p("\n[NaN RATES AFTER SPINE ALIGNMENT]")
p(f"  {'Dataset':<15} {'Overall':>9}")
p("  " + "-" * 26)
for name, df in [("prices", prices_s), ("load", load_s),
                 ("generation", generation_s), ("crossborder", crossborder_s),
                 ("weather", weather_flat), ("fuels", fuels_h)]:
    p(f"  {name:<15} {df.isna().mean().mean():>9.1%}")

# 4. Per-column NaN - only columns with >1% missing
p("\n[PER-COLUMN NaN > 1%]")
all_cols = pd.concat([generation_s, crossborder_s], axis=1)
bad_cols = all_cols.isna().mean()
bad_cols = bad_cols[bad_cols > 0.01].sort_values(ascending=False)
for col, rate in bad_cols.items():
    r = run_lengths(all_cols[col])
    max_r = int(r.max()) if len(r) > 0 else 0
    pct_long = (r > 168).mean() if len(r) > 0 else 0
    src = "gen" if col in generation_s.columns else "cb"
    p(f"  [{src}] {col:<28} NaN={rate:.1%}  max_run={max_r}h  runs>1w={pct_long:.0%}")

# 5. Cross-border FR diagnostic
p("\n[CROSS-BORDER FR DIAGNOSTIC]")
try:
    cb2 = crossborder_s.copy()
    at_ch = [c for c in cb2.columns if ('AT' in c or 'CH' in c) and 'net' not in c]
    cb2['atch'] = cb2[at_ch].abs().sum(axis=1, skipna=True)
    fr_nan2 = cb2['FR_to_DELU'].isna()
    u2, p2 = stats.mannwhitneyu(
        cb2.loc[fr_nan2, 'atch'].dropna(),
        cb2.loc[~fr_nan2, 'atch'].dropna(),
        alternative='two-sided'
    )
    verdict = 'structural reporting gap (not zero-flow)' if p2 < 0.05 else 'independent of grid activity (zero-fill candidate)'
    p(f"  FR_to_DELU NaN={fr_nan2.mean():.1%}  Mann-Whitney p={p2:.2e}")
    p(f"  Verdict: {verdict}")
except Exception as e:
    p(f"  Could not run diagnostic: {e}")

# 6. Outliers
p("\n[OUTLIERS]")
price_col2 = prices_s.columns[0]
load_col2  = load_s.columns[0]
neg_n = (prices_s[price_col2] < 0).sum()
neg_pct = neg_n / len(prices_s)
neg_min = prices_s[price_col2][prices_s[price_col2] < 0].min()
p(f"  Negative prices : {neg_n:,} hours ({neg_pct:.1%}), min={neg_min:.1f} EUR/MWh")
p(f"  Price outside [-500, 4000]: "
  f"{((prices_s[price_col2]<-500)|(prices_s[price_col2]>4000)).sum()} hours")
p(f"  Load outside [20k, 100k] MW: "
  f"{((load_s[load_col2]<20_000)|(load_s[load_col2]>100_000)).sum()} hours")
yearly_med2 = load_s[load_col2].groupby(load_s[load_col2].index.year).median()
jan1_ratios = []
for yr in range(2019, 2026):
    jan1 = load_s[load_col2][load_s[load_col2].index.normalize() == pd.Timestamp(f'{yr}-01-01', tz='UTC')]
    if yr in yearly_med2.index and len(jan1) > 0:
        jan1_ratios.append(f"{yr}:{jan1.mean()/yearly_med2[yr]:.2f}")
p(f"  New Year load dip (Jan1/yearly_median): {', '.join(jan1_ratios)}")

# 7. Combined matrix
p("\n[COMBINED MATRIX (emis_raw.parquet)]")
p(f"  Shape        : {combined.shape}")
p(f"  Date range   : {combined.index[0].date()} -> {combined.index[-1].date()}")
p(f"  Overall NaN  : {combined.isna().mean().mean():.2%}")
p(f"  Cols with NaN: {combined.isna().any().sum()} / {combined.shape[1]}")
p(f"  Clean cols   : price, load, all weather ({len(weather_flat.columns)} cols), fuels")
p(f"  Impute cols  : generation ({generation_s.shape[1]} cols), "
  f"crossborder raw ({len([c for c in crossborder_s.columns if 'net_' not in c])} cols)")

# 8. Glocal-IB interface note
p("\n[GLOCAL-IB INTERFACE]")
p("  Input : emis_raw.parquet  - float64 DataFrame, NaN = missing")
p("  Mask  : emis_mask.parquet - uint8 DataFrame,  1 = was missing")
p("  Format expected by Glocal_IB.py: [B, T, N] float32 tensor")
p("  Sliding window size (T): TBD - recommend 168 (1 week) or 720 (1 month)")
p("  Base model candidates: SAITS (already in GlocalIB/otherModel/SAITS/)")

p("\n" + "=" * 72)

report = out.getvalue()
print(report)